# 🤖 Python ML Pipeline Integration — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> An ML pipeline is a factory with three departments. The Feature Store is the parts warehouse — raw materials come in, get standardized, and sit on labeled shelves ready for any assembly line. The training pipeline is the assembly line — it picks parts from the shelf (as of yesterday, not today), assembles a model, and tests it. The serving layer is the shipping dock — takes the approved model, packages it for deployment, and sends it to customers. The key rule: training and serving must use identical feature logic or your predictions will drift from your evaluation metrics.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is ML Pipeline Integration? The Visual Model](#1) |
| 2 | [Core Concepts — Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Feature Store — Offline & Online](#5) |
| 6 | [Pattern 2: Point-in-Time Correct Feature Retrieval](#6) |
| 7 | [Pattern 3: Model Training Pipeline](#7) |
| 8 | [Pattern 4: Model Registry & A/B Testing](#8) |
| 9 | [Pattern 5: MLflow Experiment Tracking](#9) |
| 10 | [The ML Pipeline Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. What Is ML Pipeline Integration? The Visual Model

---

```
ML PIPELINE ARCHITECTURE

  RAW DATA                FEATURE ENGINEERING       MODEL
  ────────                ───────────────────       ─────
  orders_fact   ──────►  feature_store (offline) ──► training job
  user_events   ──────►  (Parquet on S3)         ──► model artifact
  payments      ──────►                           ──► model registry
                                                       │
  SERVING                                         ──► staging → production
  ───────                                              │
  API request   ──────►  feature_store (online)  ──► model.predict()
  user_id=123   ──────►  (Redis / DynamoDB)       ──► score → response

TRAINING / SERVING SKEW (the big problem):
  Training:  feature X computed as mean(last 30 days orders)
  Serving:   feature X computed as mean(last 7 days orders) [different code!]
  Result:    model was trained on different distribution → predictions degrade
  Fix:       ONE feature computation function used by BOTH pipelines

POINT-IN-TIME CORRECTNESS:
  Training: for label at 2024-03-15, use features as of 2024-03-14
            NOT features from 2024-03-16 (that's data leakage!)
  Example:  predicting churn at 2024-03-15 should use clicks up to 2024-03-14
            If you include 2024-03-15 clicks, you're leaking the future into training

MLFLOW CONCEPTS:
  Experiment:  named collection of runs (e.g. 'churn_model_v2')
  Run:         one model training attempt with params + metrics + artifacts
  Artifact:    saved model file, feature importance plot, etc.
  Model Registry: versioned catalog of production-approved models
  Stage:       None → Staging → Production → Archived
```


<a id='2'></a>
## 2. Core Concepts — Setup

In [ ]:
import random
import math
import hashlib
import json
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable
from collections import defaultdict
import time
import copy

random.seed(42)

# simulate raw event data (user clickstream)
def make_user_events(n_users=500, days=60):
    events = []
    for uid in range(1, n_users + 1):
        for day in range(days):
            date = f'2024-{(day//28)+1:02d}-{(day%28)+1:02d}'
            if random.random() > 0.3:  # user active 70% of days
                n_clicks = random.randint(1, 20)
                events.append({'user_id': uid, 'date': date,
                               'clicks': n_clicks,
                               'purchases': random.randint(0, 3),
                               'revenue': round(random.uniform(0, 200), 2)})
    return events

USER_EVENTS = make_user_events()
print(f"User events: {len(USER_EVENTS):,} rows for {500} users over 60 days")

# group by user for easy feature computation
EVENTS_BY_USER: Dict[int, List[Dict]] = defaultdict(list)
for e in USER_EVENTS:
    EVENTS_BY_USER[e['user_id']].append(e)

# simulate churn labels (1 = churned within 30 days after label_date)
CHURN_LABELS = {uid: random.random() < 0.2 for uid in range(1, 501)}  # 20% churn rate
print(f"Churn rate: {sum(CHURN_LABELS.values())/len(CHURN_LABELS):.1%}")
print("Setup complete.")

<a id='3'></a>
## 3. The Core API — All Operations

---

```
ML PIPELINE OPERATIONS
────────────────────────────────────────────────────────────────────────────
OPERATION                          WHAT IT DOES
────────────────────────────────────────────────────────────────────────────
feature_store.compute(entity, date)  compute features for entity as of date
feature_store.get_online(entity)     fetch latest features from fast store
feature_store.get_historical(df)     point-in-time correct feature join
training_pipeline.run(label_date)    extract features + labels → train model
model_registry.register(model, name) save model artifact with version
model_registry.promote(name, stage)  move version to Staging/Production
model.predict(features)              score one or batch of feature vectors
ab_test.assign(user_id)             assign user to control or treatment
ab_test.log_outcome(user_id, metric) record outcome for significance test
mlflow.start_run()                   begin tracking a training run
mlflow.log_param(key, value)         record hyperparameter
mlflow.log_metric(key, value, step)  record training metric
mlflow.log_artifact(path)            save model/plot to run storage
────────────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  Use future features in training (data leakage) — always as_of date
❌  Different feature code in training vs serving (training/serving skew)
❌  Deploy model without comparing to production baseline
❌  Use same data for training + evaluation (overfitting / overestimated metrics)
❌  Skip feature versioning — re-running training must reproduce same result
❌  Store model artifacts in local files without registry — can't track versions
```


In [ ]:
# Core API demo: feature computation and usage

def compute_user_features(user_id, events, as_of_date):
    # ALL feature computation goes here — same function for training and serving
    # NEVER duplicate this logic in a separate serving path!
    user_events = [e for e in events if e['date'] < as_of_date]  # as_of_date exclusive
    if not user_events:
        return None
    recent = [e for e in user_events if e['date'] >= as_of_date[:7] + '-01']  # this month
    return {
        'user_id':           user_id,
        'total_clicks_30d':  sum(e['clicks'] for e in user_events[-30:]),
        'total_revenue_30d': sum(e['revenue'] for e in user_events[-30:]),
        'avg_daily_clicks':  sum(e['clicks'] for e in user_events) / max(len(user_events), 1),
        'days_active':       len(user_events),
        'recent_purchases':  sum(e['purchases'] for e in recent),
        'as_of_date':        as_of_date,
    }

# demo: compute features for a few users
print("=== Feature computation (as of 2024-03-01) ===")
for uid in [1, 2, 3]:
    feats = compute_user_features(uid, EVENTS_BY_USER[uid], '2024-03-01')
    if feats:
        print(f"  user_{uid}: clicks_30d={feats['total_clicks_30d']} revenue_30d=${feats['total_revenue_30d']:.2f} days_active={feats['days_active']}")

print()
print("=== Training/Serving Skew Demo ===")
# correct (same function, same logic)
train_feat  = compute_user_features(1, EVENTS_BY_USER[1], '2024-03-01')
serve_feat  = compute_user_features(1, EVENTS_BY_USER[1], '2024-03-15')
print(f"  Training (as_of 2024-03-01):  clicks={train_feat['total_clicks_30d']}")
print(f"  Serving  (as_of 2024-03-15):  clicks={serve_feat['total_clicks_30d']}")
print(f"  Same function → drift is real data drift, not code skew  ✅")

print("\nCore API demo complete.")

<a id='4'></a>
## 4. Decision Map — When To Use What

---

```
SIGNAL IN THE PROBLEM                        WHAT TO DO
────────────────────────────────────────────────────────────────────────────
Training and serving give different results  Eliminate training/serving skew
Model leaks future info                      Point-in-time correct joins
Features reused across multiple models       Feature Store (offline layer)
Low-latency serving (<10ms)                  Feature Store (online: Redis/DynamoDB)
Track hyperparameters + metrics              MLflow experiments
Deploy new model safely                      A/B test + model registry staging
Model degrades over time                     Data/concept drift monitoring
Multiple model versions in production        Model registry + gradual rollout
Batch scoring 1M users nightly               Offline feature join + batch predict
Real-time scoring per API request            Online feature fetch + predict
────────────────────────────────────────────────────────────────────────────
```


<a id='5'></a>
## 5. 🧩 Pattern 1: Feature Store — Offline & Online

---

```
PROBLEM:
  3 different ML teams compute the same 'user_30d_revenue' feature differently.
  Each team has a different bug. Training fails to match production predictions.
  How do you centralize feature computation and serve it at low latency?

APPROACH:
  Feature Store = centralized feature computation + storage in two layers:
  OFFLINE: historical features (Parquet on S3) — for training
  ONLINE:  latest features (Redis/DynamoDB) — for real-time serving

OFFLINE STORE:
  Storage: S3/Delta partitioned by entity_id and date
  Use case: point-in-time correct training dataset generation
  Latency: seconds to minutes (batch reads)

ONLINE STORE:
  Storage: Redis/DynamoDB keyed by entity_id
  Use case: real-time serving during inference API calls
  Latency: < 10ms

MATERIALISATION PIPELINE:
  daily job: compute features for all entities as of today
  → write to offline store (Parquet partition for today)
  → write to online store (overwrite Redis key per entity)

SLOW MOTION: serving flow
  API request: {user_id: 123, event: 'loan_application'}
  Step 1: fetch user_id=123 from online store → {clicks_30d: 450, revenue_30d: 1200}
  Step 2: append request context features → {time_of_day: 'evening', device: 'mobile'}
  Step 3: concatenate → feature vector → model.predict() → {score: 0.72, label: HIGH_RISK}
  Step 4: return response in < 50ms

KEY INSIGHT:
  The feature computation logic is written once and deployed everywhere.
  Offline store = bulk historical reads. Online store = single-key fast reads.
  Feast, Tecton, Hopsworks are purpose-built feature store platforms.

TIME / SPACE:
  Materialise offline: O(N × F) — N entities × F features
  Materialise online:  O(N) — write one Redis key per entity
  Serve online:        O(1) — single key lookup
```


In [ ]:
# Pattern 1: Feature Store — offline and online layers

# Slow motion: materialise features for all users as of today
# step 1: for each user, call compute_user_features(uid, events, today)
# step 2: write to offline store (keyed by user_id + date)
# step 3: write latest to online store (keyed by user_id only)
# step 4: serving path: online_store.get(user_id) → features in O(1)

class OfflineFeatureStore:
    """
    ML Pipeline Pattern 1 — Offline feature store (training data).
    Approach: Store historical features partitioned by entity + date.
    Time:  O(N) lookup for one entity's history
    Space: O(N × D) — N entities × D dates of history
    """
    def __init__(self):
        # keyed by (entity_id, date) → feature dict
        self._store: Dict[tuple, dict] = {}

    def write(self, entity_id, date, features):
        self._store[(entity_id, date)] = dict(features)

    def get(self, entity_id, date) -> Optional[dict]:
        return self._store.get((entity_id, date))

    def get_history(self, entity_id, start_date, end_date) -> List[dict]:
        return [v for (eid, d), v in self._store.items()
                if eid == entity_id and start_date <= d <= end_date]

class OnlineFeatureStore:
    """
    ML Pipeline Pattern 1 — Online feature store (real-time serving).
    Approach: Redis-like dict keyed by entity_id; single O(1) lookup at serving time.
    Time:  O(1) get/write — hash map lookup
    Space: O(N) — N entities with latest features only
    """
    def __init__(self):
        self._store: Dict[Any, dict] = {}  # entity_id → latest features

    def write(self, entity_id, features):
        self._store[entity_id] = dict(features)

    def get(self, entity_id) -> Optional[dict]:
        return self._store.get(entity_id)

class FeatureStoreMaterialiser:
    def __init__(self, offline: OfflineFeatureStore, online: OnlineFeatureStore):
        self.offline = offline
        self.online  = online

    def materialise(self, all_events, as_of_date, sample_users=None):
        users = sample_users or list(EVENTS_BY_USER.keys())
        written = 0
        for uid in users:
            feats = compute_user_features(uid, all_events.get(uid, []), as_of_date)
            if feats:
                self.offline.write(uid, as_of_date, feats)
                self.online.write(uid, feats)  # always latest
                written += 1
        return written

offline_store = OfflineFeatureStore()
online_store  = OnlineFeatureStore()
materialiser  = FeatureStoreMaterialiser(offline_store, online_store)

print("=== Materialising Features for 100 Users ===")
dates = ['2024-01-15', '2024-02-15', '2024-03-15']
for d in dates:
    n = materialiser.materialise(EVENTS_BY_USER, d, sample_users=list(range(1, 101)))
    print(f"  {d}: {n} users materialised")

print()
print("=== Offline Store: historical feature retrieval ===")
uid_sample = 42
history = offline_store.get_history(uid_sample, '2024-01-01', '2024-12-31')
print(f"  User {uid_sample}: {len(history)} historical feature snapshots")
for h in history:
    print(f"    {h['as_of_date']}: clicks={h['total_clicks_30d']} revenue=${h['total_revenue_30d']:.2f}")

print()
print("=== Online Store: real-time serving ===")
start = time.perf_counter()
features = online_store.get(uid_sample)
latency  = (time.perf_counter() - start) * 1000
print(f"  User {uid_sample} features: {features}")
print(f"  Lookup latency: {latency:.3f}ms (O(1) hash map)")

print("\nFeature Store pattern complete.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Point-in-Time Correct Feature Retrieval

---

```
PROBLEM:
  You want to train a churn model. Labels are: did user churn in the 30 days
  AFTER 2024-03-01? Features should be behavior BEFORE 2024-03-01.
  If you accidentally use behavior from 2024-03-15 in features for the
  2024-03-01 label — that's data leakage. Model looks great on evaluation
  but fails completely in production (it saw the future).

APPROACH:
  For each (entity, label_date), retrieve features as_of (label_date - 1 day).
  Never allow feature_date >= label_date.

SLOW MOTION: training dataset construction
  labels = [(user_1, 2024-03-01, churned=1),
            (user_2, 2024-03-01, churned=0), ...]

  For (user_1, label_date=2024-03-01):
    feature_as_of = 2024-02-29  ← one day BEFORE label date
    features = offline_store.get(user_1, '2024-02-29')
    row = {**features, 'label': 1}

  WRONG (data leakage):
    features = offline_store.get(user_1, '2024-03-15')  ← future features!
    model learns from user's behavior AFTER the event it's predicting

FEATURE FRESHNESS:
  If no snapshot exists for exact as_of date → use most recent BEFORE that date
  This is the 'point-in-time join' in feature store terminology

KEY INSIGHT:
  Data leakage is invisible in offline evaluation — metrics look great.
  It only appears in production where you can't see the future.
  The discipline is: ALWAYS compute as_of = label_date - 1 day (or more).

TIME / SPACE:
  PIT join: O(N × log(D)) — N entities × binary search over D date history
  Dataset construction: O(N × F) — N label rows × F features per row
```


In [ ]:
# Pattern 2: Point-in-time correct feature retrieval

# Slow motion: training dataset construction
# step 1: for each (user, label_date), compute as_of = label_date - 1 day
# step 2: retrieve features as_of that date (no future data)
# step 3: join features + label → training row
# step 4: verify: no feature column has data from after label_date

def date_subtract_days(date_str, days):
    year, month, day = map(int, date_str.split('-'))
    day -= days
    if day <= 0:
        month -= 1
        if month <= 0:
            month = 12; year -= 1
        day += 28  # simplified
    return f'{year}-{month:02d}-{day:02d}'

def build_training_dataset(label_rows, offline_store, lookback_days=1):
    """
    Point-in-time correct training dataset construction.
    Args:
        label_rows (list of dict): each has user_id, label_date, label
        offline_store (OfflineFeatureStore): feature history
        lookback_days (int): how many days before label_date to use as feature cutoff
    Returns:
        List of training rows with features + label
    """
    training_rows = []
    leakage_check_pass = 0
    leakage_check_fail = 0

    for row in label_rows:
        uid        = row['user_id']
        label_date = row['label_date']
        label      = row['label']

        # point-in-time cutoff: features must be from BEFORE label_date
        feature_as_of = date_subtract_days(label_date, lookback_days)

        # get features from offline store at the correct point in time
        features = offline_store.get(uid, feature_as_of)
        if features is None:
            # fall back to most recent feature snapshot before as_of
            history = offline_store.get_history(uid, '2024-01-01', feature_as_of)
            features = sorted(history, key=lambda f: f['as_of_date'])[-1] if history else None

        if features is None:
            continue  # skip users with no feature history

        # leakage check: verify feature date < label date
        if features['as_of_date'] < label_date:
            leakage_check_pass += 1
        else:
            leakage_check_fail += 1
            continue  # reject row — potential leakage

        training_row = {k: v for k, v in features.items() if k != 'as_of_date'}
        training_row['label'] = label
        training_rows.append(training_row)

    return training_rows, leakage_check_pass, leakage_check_fail

# create label rows — churn labels at 2024-03-01
label_rows = [
    {'user_id': uid, 'label_date': '2024-03-15', 'label': int(CHURN_LABELS[uid])}
    for uid in range(1, 101)  # 100 users
]

print("=== Building Point-in-Time Correct Training Dataset ===")
print("  Label date: 2024-03-15  (churn in 30 days after this date)")
print("  Feature as_of: 2024-03-14  (features BEFORE label date)")
print()

training_data, passed, failed = build_training_dataset(label_rows, offline_store)
print(f"  Total label rows:   {len(label_rows)}")
print(f"  Training rows built: {len(training_data)}")
print(f"  Leakage checks passed: {passed}  failed/rejected: {failed}")

if training_data:
    print()
    print("=== Sample Training Row ===")
    sample = training_data[0]
    for k, v in sample.items():
        print(f"  {k:25s}: {v}")

print()
# demonstrate leakage scenario
print("=== Data Leakage Demo (WRONG — using future features) ===")
leaky_row = label_rows[0]
future_features = offline_store.get(leaky_row['user_id'], '2024-03-15')  # same date as label!
if future_features:
    print(f"  Features as_of=2024-03-15 (= label_date): {future_features['as_of_date']}")
    print(f"  This is LEAKAGE — features include the very day we're predicting!")
    print(f"  Model will overfit → great evaluation metrics, poor production performance")

print("\nPoint-in-time feature retrieval pattern complete.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Model Training Pipeline

---

```
PROBLEM:
  Build a reproducible training pipeline that: pulls features,
  splits data, trains a model, evaluates it, and produces metrics.

APPROACH:
  1. Feature retrieval:  point-in-time correct (Pattern 2)
  2. Train/test split:   temporal split (not random — avoids leakage)
  3. Train model:        fit on train, evaluate on test
  4. Evaluate:           accuracy, AUC, precision, recall
  5. Log to MLflow:      params, metrics, model artifact

TEMPORAL SPLIT (vs random split):
  Random split:   train=[Mar1, Feb5, Mar10, ...] test=[Mar3, Mar7, ...]
  Temporal split: train=[all before cutoff] test=[all after cutoff]
  Reason:         in production, model always predicts for future events
                  random split leaks future events into training set

SIMPLE MODEL (logistic regression from scratch):
  sigmoid(x) = 1 / (1 + e^(-x))
  prediction = sigmoid(w·x + b)
  gradient descent update:
    error = label - prediction
    w += lr × error × x
    b += lr × error

EVALUATION METRICS:
  Accuracy:  correct / total (misleading for imbalanced labels)
  Precision: TP / (TP + FP)  (when positive label is expensive to flag wrong)
  Recall:    TP / (TP + FN)  (when missing a positive is costly)
  AUC:       area under ROC curve (threshold-independent)

KEY INSIGHT:
  Always use temporal split, not random split, for time-series predictions.
  Log everything to an experiment tracker — reproducibility is non-negotiable.

TIME / SPACE:
  Training:   O(N × F × epochs) — gradient descent
  Evaluation: O(N_test)
  Space:      O(F) — model weights (one per feature)
```


In [ ]:
# Pattern 3: Model training pipeline

# Slow motion: logistic regression from scratch on churn data
# step 1: extract feature matrix X and label vector y from training_data
# step 2: normalize features (mean=0, std=1)
# step 3: gradient descent for 100 epochs
# step 4: evaluate on test set → accuracy, precision, recall

FEATURE_COLS = ['total_clicks_30d', 'total_revenue_30d', 'avg_daily_clicks',
                'days_active', 'recent_purchases']

def sigmoid(x):
    x = max(-500, min(500, x))  # clip to avoid overflow
    return 1.0 / (1.0 + math.exp(-x))

class SimpleLogisticRegression:
    """
    ML Pipeline Pattern 3 — Logistic regression from scratch.
    Approach: Gradient descent with sigmoid activation for binary classification.
    Time:  O(N × F × epochs)
    Space: O(F) weights
    """
    def __init__(self, lr=0.01, epochs=100):
        self.lr = lr; self.epochs = epochs
        self.weights = None; self.bias = 0.0
        self.feature_means = None; self.feature_stds = None

    def _normalize(self, X, fit=False):
        if fit:
            self.feature_means = [sum(row[i] for row in X)/len(X) for i in range(len(X[0]))]
            self.feature_stds  = [max(math.sqrt(sum((r[i]-self.feature_means[i])**2 for r in X)/len(X)), 1e-8)
                                   for i in range(len(X[0]))]
        return [[(x - m)/s for x, m, s in zip(row, self.feature_means, self.feature_stds)]
                for row in X]

    def fit(self, X_train, y_train):
        X = self._normalize(X_train, fit=True)
        n_features = len(X[0])
        self.weights = [0.0] * n_features
        for epoch in range(self.epochs):
            for xi, yi in zip(X, y_train):
                pred  = sigmoid(sum(w*x for w, x in zip(self.weights, xi)) + self.bias)
                error = yi - pred
                self.weights = [w + self.lr * error * x for w, x in zip(self.weights, xi)]
                self.bias   += self.lr * error
        return self

    def predict_proba(self, X):
        X_norm = self._normalize(X)
        return [sigmoid(sum(w*x for w,x in zip(self.weights, xi)) + self.bias) for xi in X_norm]

    def predict(self, X, threshold=0.5):
        return [1 if p >= threshold else 0 for p in self.predict_proba(X)]

def evaluate(y_true, y_pred):
    tp = sum(1 for a,p in zip(y_true, y_pred) if a==1 and p==1)
    tn = sum(1 for a,p in zip(y_true, y_pred) if a==0 and p==0)
    fp = sum(1 for a,p in zip(y_true, y_pred) if a==0 and p==1)
    fn = sum(1 for a,p in zip(y_true, y_pred) if a==1 and p==0)
    acc  = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn) else 0
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec  = tp/(tp+fn) if (tp+fn) else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'tp':tp,'fp':fp,'tn':tn,'fn':fn}

# build feature matrix from training_data
X_all = [[row.get(c, 0.0) for c in FEATURE_COLS] for row in training_data]
y_all = [row['label'] for row in training_data]

# temporal split: first 70% train, last 30% test
split = int(len(X_all) * 0.70)
X_train, y_train = X_all[:split], y_all[:split]
X_test,  y_test  = X_all[split:], y_all[split:]

print(f"Dataset: {len(X_all)} rows, {len(FEATURE_COLS)} features")
print(f"Churn rate: {sum(y_all)/len(y_all):.1%}")
print(f"Train: {len(X_train)} rows  Test: {len(X_test)} rows")

model = SimpleLogisticRegression(lr=0.05, epochs=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
metrics = evaluate(y_test, y_pred)
print(f"\n=== Model Evaluation ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:12s}: {v:.3f}")
    else:
        print(f"  {k:12s}: {v}")

print(f"\nFeature weights:")
for col, w in zip(FEATURE_COLS, model.weights):
    print(f"  {col:25s}: {w:+.4f}")

print("\nModel training pipeline complete.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Model Registry & A/B Testing

---

```
PROBLEM:
  You trained model_v2 with better evaluation metrics than model_v1.
  How do you safely deploy it to production without risking a regression?

APPROACH:
  1. Model Registry: version-controlled store of approved model artifacts
  2. Stages: None → Staging → Production → Archived
  3. A/B test: split traffic — 10% model_v2, 90% model_v1
  4. Promote: after statistical significance achieved, promote v2 to 100%

MODEL REGISTRY WORKFLOW:
  mlflow.register_model(run_id, name='churn_model')  → version 1
  client.transition_model_version_stage(name, 1, 'Staging')  → test
  run shadow mode: score real traffic with v1 (production) and v2 (shadow)
  compare metrics → if v2 wins → promote to Production → v1 → Archived

A/B TEST DESIGN:
  Assignment: hash(user_id) % 100 < 10 → treatment (v2)
              hash(user_id) % 100 >= 10 → control (v1)
  Metrics:    click-through rate, conversion rate, churn rate (7-day)
  Significance: Chi-square test for proportions, t-test for means
  Sample size: minimum detectable effect = 5% lift → N ≈ 1600 per group

SLOW MOTION: A/B test outcome
  control (v1, 90%): 1000 users, 200 churned → churn_rate = 20%
  treatment (v2, 10%): 100 users, 15 churned → churn_rate = 15%
  relative improvement = (20% - 15%) / 20% = 25% lift
  test for significance → p < 0.05 → promote v2

KEY INSIGHT:
  Never evaluate a new model only on offline metrics.
  Real A/B test catches distribution shift and user behavior differences
  that offline holdout evaluation misses.

TIME / SPACE:
  A/B assignment: O(1) — deterministic hash
  Significance test: O(N) — iterate over recorded outcomes
```


In [ ]:
# Pattern 4: Model registry and A/B testing

# Slow motion: deterministic A/B assignment + outcome recording
# step 1: hash user_id → 0-99 bucket → assign to control or treatment
# step 2: each user's request routed to correct model version
# step 3: record outcome (churned: yes/no) per group
# step 4: compare rates → check for significance

@dataclass
class ModelVersion:
    name:       str
    version:    int
    run_id:     str
    stage:      str  # 'None', 'Staging', 'Production', 'Archived'
    metrics:    Dict[str, float] = field(default_factory=dict)
    model:      Any = None

class ModelRegistry:
    def __init__(self):
        self.models: Dict[str, List[ModelVersion]] = defaultdict(list)

    def register(self, name, model, metrics, run_id):
        versions = self.models[name]
        version  = len(versions) + 1
        mv = ModelVersion(name=name, version=version, run_id=run_id,
                          stage='None', metrics=metrics, model=model)
        versions.append(mv)
        print(f"  Registered {name} v{version} | metrics={metrics}")
        return mv

    def transition(self, name, version, new_stage):
        mv = self.models[name][version - 1]
        old_stage = mv.stage
        mv.stage  = new_stage
        print(f"  {name} v{version}: {old_stage} → {new_stage}")

    def get_production(self, name) -> Optional[ModelVersion]:
        return next((mv for mv in reversed(self.models[name]) if mv.stage == 'Production'), None)

class ABTestRouter:
    def __init__(self, control_model, treatment_model, treatment_pct=10):
        self.control   = control_model
        self.treatment = treatment_model
        self.pct       = treatment_pct
        self.outcomes  = {'control': [], 'treatment': []}

    def assign(self, user_id) -> str:
        # deterministic: same user always gets same assignment
        bucket = int(hashlib.md5(str(user_id).encode()).hexdigest(), 16) % 100
        return 'treatment' if bucket < self.pct else 'control'

    def score(self, user_id, features):
        group = self.assign(user_id)
        model = self.treatment if group == 'treatment' else self.control
        score = model.predict_proba([features])[0] if model else random.uniform(0, 1)
        return group, score

    def record_outcome(self, user_id, churned):
        group = self.assign(user_id)
        self.outcomes[group].append(churned)

    def report(self):
        for group, outcomes in self.outcomes.items():
            if not outcomes: continue
            rate = sum(outcomes) / len(outcomes)
            print(f"  {group:12s}: n={len(outcomes):4d} churn_rate={rate:.1%}")

# train two model versions
model_v1 = SimpleLogisticRegression(lr=0.05, epochs=100).fit(X_train, y_train)
model_v2 = SimpleLogisticRegression(lr=0.05, epochs=300).fit(X_train, y_train)  # more training

m1_metrics = evaluate(y_test, model_v1.predict(X_test))
m2_metrics = evaluate(y_test, model_v2.predict(X_test))

registry = ModelRegistry()
print("=== Model Registry ===")
mv1 = registry.register('churn_model', model_v1, {'f1': round(m1_metrics['f1'],3)}, 'run_001')
mv2 = registry.register('churn_model', model_v2, {'f1': round(m2_metrics['f1'],3)}, 'run_002')
registry.transition('churn_model', 1, 'Production')
registry.transition('churn_model', 2, 'Staging')

print()
print("=== A/B Test: v1 (90%) vs v2 (10%) ===")
ab = ABTestRouter(model_v1, model_v2, treatment_pct=10)

# simulate 500 users going through scoring + real outcome recorded
for uid in range(1, 501):
    feats = [random.uniform(0,1) for _ in FEATURE_COLS]  # simplified
    group, score = ab.score(uid, feats)
    # real outcome (independent of model)
    actual_churn = CHURN_LABELS.get(uid, 0)
    ab.record_outcome(uid, actual_churn)

ab.report()

print()
# simple significance check (proportions z-test)
ctrl = ab.outcomes['control']
trt  = ab.outcomes['treatment']
p1 = sum(ctrl)/len(ctrl)
p2 = sum(trt)/len(trt)
pooled_p = (sum(ctrl)+sum(trt))/(len(ctrl)+len(trt))
se = math.sqrt(pooled_p*(1-pooled_p)*(1/len(ctrl)+1/len(trt))) + 1e-10
z = abs(p1-p2)/se
significant = z > 1.96  # 95% confidence
print(f"  z-score: {z:.2f}  significant (p<0.05): {significant}")
if significant:
    winner = 'treatment (v2)' if p2 < p1 else 'control (v1)'
    print(f"  Winner: {winner}")
    registry.transition('churn_model', 2, 'Production')
    registry.transition('churn_model', 1, 'Archived')

print("\nModel registry and A/B test pattern complete.")

<a id='9'></a>
## 9. 🧩 Pattern 5: MLflow Experiment Tracking

---

```
PROBLEM:
  You run 50 training experiments tuning hyperparameters.
  How do you track which params produced which metrics,
  and reproduce the best run 3 months later?

APPROACH:
  MLflow: log params, metrics, artifacts per run.
  Compare runs in UI. Register best model.
  Everything stored: git commit, params, metrics, model file.

MLFLOW CONCEPTS:
  Experiment:  named bucket of related runs (e.g. 'churn_q1_2024')
  Run:         single training attempt — has params + metrics + artifacts
  Param:       hyperparameter (lr=0.01, epochs=100, feature_set='v2')
  Metric:      scalar performance number, can be logged per step (epoch)
  Artifact:    file output (model.pkl, feature_importance.png, data.parquet)

AUTOLOG:
  mlflow.sklearn.autolog()  → automatically logs all sklearn params + metrics
  mlflow.spark.autolog()    → logs Spark model params
  Reduces boilerplate for standard ML frameworks

SLOW MOTION: hyperparameter search
  for lr in [0.001, 0.01, 0.1]:
    for epochs in [50, 100, 200]:
      with mlflow.start_run():      ← start tracking
        mlflow.log_param('lr', lr)
        mlflow.log_param('epochs', epochs)
        model.fit(X_train, y_train)
        f1 = evaluate(y_test, model.predict(X_test))['f1']
        mlflow.log_metric('f1', f1)
        mlflow.sklearn.log_model(model, 'model')
  best_run = client.search_runs(filter='metrics.f1 > 0.85', order_by='f1 DESC')

KEY INSIGHT:
  Without experiment tracking, ML experiments are irreproducible.
  MLflow makes every experiment a first-class artifact — queryable, comparable, auditable.

TIME / SPACE:
  Log per run: O(1) per metric/param step
  Search runs: O(R × M) — R runs × M metrics
  Artifact storage: O(model_size) per run — use artifact store (S3)
```


In [ ]:
# Pattern 5: MLflow experiment tracking simulation

# Slow motion: hyperparameter sweep with run tracking
# step 1: start run → log params (lr, epochs, feature_set)
# step 2: train model → log metric per epoch (loss)
# step 3: evaluate → log final metrics (f1, accuracy)
# step 4: save model artifact → register in Model Registry
# step 5: query best run → promote to production

@dataclass
class MLflowRun:
    run_id:    str
    params:    Dict[str, Any] = field(default_factory=dict)
    metrics:   Dict[str, Any] = field(default_factory=dict)  # name → list of values
    artifacts: List[str] = field(default_factory=list)
    status:    str = 'RUNNING'

class MLflowTracker:
    """
    ML Pipeline Pattern 5 — Experiment tracking simulation.
    Approach: Log params, metrics, artifacts per run; query best run.
    Time:  O(R × M) for searching R runs with M metrics
    Space: O(R × (P + M + A)) — runs × (params + metrics + artifacts)
    """
    def __init__(self, experiment_name):
        self.experiment = experiment_name
        self.runs: List[MLflowRun] = []
        self._active: Optional[MLflowRun] = None

    def start_run(self, run_name=None):
        run_id = hashlib.md5(f'{time.time()}{run_name}'.encode()).hexdigest()[:8]
        self._active = MLflowRun(run_id=run_id)
        self.runs.append(self._active)
        return run_id

    def log_param(self, key, value):
        self._active.params[key] = value

    def log_metric(self, key, value, step=None):
        if key not in self._active.metrics:
            self._active.metrics[key] = []
        self._active.metrics[key].append((step, value))

    def log_artifact(self, path):
        self._active.artifacts.append(path)

    def end_run(self, status='FINISHED'):
        self._active.status = status
        self._active = None

    def get_best_run(self, metric, higher_is_better=True) -> Optional[MLflowRun]:
        finished = [r for r in self.runs if r.status == 'FINISHED' and metric in r.metrics]
        if not finished:
            return None
        def final_metric(run):
            vals = run.metrics[metric]
            return vals[-1][1] if vals else 0
        return sorted(finished, key=final_metric, reverse=higher_is_better)[0]

    def print_runs(self):
        print(f"  {'run_id':10s} {'lr':6s} {'epochs':8s} {'f1':6s} {'accuracy':10s}")
        for r in self.runs:
            lr  = r.params.get('lr', '-')
            ep  = r.params.get('epochs', '-')
            f1  = r.metrics.get('f1', [(0,0)])[-1][1]
            acc = r.metrics.get('accuracy', [(0,0)])[-1][1]
            print(f"  {r.run_id:10s} {str(lr):6s} {str(ep):8s} {f1:.3f}  {acc:.3f}")

tracker = MLflowTracker('churn_model_q1_2024')

# hyperparameter sweep
print("=== Hyperparameter Sweep ===")
param_grid = [
    {'lr': 0.001, 'epochs': 50},
    {'lr': 0.01,  'epochs': 100},
    {'lr': 0.05,  'epochs': 200},
    {'lr': 0.1,   'epochs': 50},
    {'lr': 0.05,  'epochs': 100},
]

for params in param_grid:
    run_id = tracker.start_run()
    tracker.log_param('lr', params['lr'])
    tracker.log_param('epochs', params['epochs'])
    tracker.log_param('feature_set', 'v2')

    # train
    m = SimpleLogisticRegression(lr=params['lr'], epochs=params['epochs'])
    m.fit(X_train, y_train)

    # evaluate
    preds = m.predict(X_test)
    mets  = evaluate(y_test, preds)

    for step in range(0, params['epochs'], 10):
        # simulate per-epoch metric
        tracker.log_metric('loss', random.uniform(0.3, 0.8) - step/params['epochs']*0.3, step)

    tracker.log_metric('f1',        mets['f1'])
    tracker.log_metric('accuracy',  mets['accuracy'])
    tracker.log_metric('precision', mets['precision'])
    tracker.log_metric('recall',    mets['recall'])
    tracker.log_artifact(f'model_lr{params["lr"]}_ep{params["epochs"]}.pkl')
    tracker.end_run()

tracker.print_runs()

print()
best = tracker.get_best_run('f1')
print(f"=== Best Run ===")
print(f"  run_id: {best.run_id}")
print(f"  params: {best.params}")
print(f"  f1:     {best.metrics['f1'][-1][1]:.3f}")
print(f"  artifact: {best.artifacts[0]}")

print("\nMLflow tracking pattern complete.")

<a id='10'></a>
## 10. The ML Pipeline Decision Map

---

```
PROBLEM                                    PATTERN        TOOL
────────────────────────────────────────────────────────────────────────────
Features reused across models              Feature Store  Feast, Tecton, Hopsworks
Training vs serving predictions differ     PIT correct    Feature Store offline
Need low-latency feature retrieval         Online store   Redis, DynamoDB
Track hyperparameter experiments           MLflow         experiments/runs
Deploy new model safely                    Model Registry MLflow Registry
Compare v1 vs v2 in production             A/B test       hash-based assignment
Model degrades over time                   Drift monitor  Evidently, Arize
Batch scoring 1M users nightly             Offline predict Spark + offline store
Real-time scoring per API call             Online predict  FastAPI + online store
────────────────────────────────────────────────────────────────────────────

TRAINING/SERVING SKEW PREVENTION:
  Rule 1: one feature computation function, used by both pipelines
  Rule 2: feature store enforces point-in-time correctness
  Rule 3: log feature distributions at training AND serving — compare

MODEL PROMOTION CHECKLIST:
  □ Offline evaluation: F1 / AUC better than baseline?
  □ Shadow mode: score traffic without serving results → compare
  □ A/B test: statistically significant improvement in production metric?
  □ Latency: p99 serving latency < SLA (e.g., < 100ms)?
  □ Fairness: no unexpected demographic disparities?
  □ Model card: document training data, metrics, known limitations
```


<a id='11'></a>
## 11. Interview Cheat Sheet

---

### When to reach for each pattern:

| Signal | Pattern |
|--------|----------|
| "Share features across models" | Feature Store |
| "Training/serving mismatch" | PIT correct join |
| "Track experiments" | MLflow |
| "Safe deployment" | Model Registry + A/B test |
| "Real-time scoring" | Online feature store |

---

### Key concepts — memorize these:

```
Training/serving skew:  different feature code in train vs serve → wrong predictions
Data leakage:           using features from t>=label_date to predict label at t
PIT join:               features.as_of_date < label_date — always enforced
Feature freshness:      how stale can online features be? (SLA for materialisation)
Model stages:           None → Staging → Production → Archived
A/B assignment:         hash(user_id) % 100 < pct → deterministic, stable
```

---

### Common templates:

```python
# TEMPLATE: MLflow tracking
with mlflow.start_run():
    mlflow.log_param('lr', 0.01)
    model.fit(X_train, y_train)
    mlflow.log_metric('f1', f1_score(y_test, model.predict(X_test)))
    mlflow.sklearn.log_model(model, 'model')

# TEMPLATE: A/B assignment
def get_model(user_id, pct_treatment=10):
    bucket = int(hashlib.md5(str(user_id).encode()).hexdigest(), 16) % 100
    return model_v2 if bucket < pct_treatment else model_v1

# TEMPLATE: PIT join guard
assert feature_as_of_date < label_date, 'Data leakage detected!'
features = offline_store.get(entity_id, feature_as_of_date)
```

---

### Gotchas to not forget:

```
❌  Random train/test split on time-series data — leaks future into training
❌  Duplicate feature computation code in training and serving pipelines
❌  Deploying without A/B test — offline metrics don't guarantee production lift
❌  Online store with stale features — materialisation lag > feature freshness SLA
✅  One feature function, two pipelines (training + serving) — the golden rule
✅  Log feature distributions at serving time — compare to training distribution
✅  Model registry stages = audit trail of what was in production and when
✅  A/B test with deterministic hash = reproducible, consistent user experience
```


<a id='12'></a>
## 12. Summary Map

---

```
                   🤖 ML PIPELINE INTEGRATION
                              │
         ┌────────────────────┼────────────────────┐
         │                    │                    │
   FEATURE LAYER         TRAINING              SERVING
   (Patterns 1,2)        (Pattern 3)           (Patterns 4,5)
         │                    │                    │
  Feature Store          Temporal split      Model Registry
  Offline (S3/Parquet)   Logistic/tree       Stage: None→Prod
  Online (Redis)         Evaluate: F1, AUC   A/B test (hash)
  PIT correct join       Avoid leakage       MLflow tracking
  One compute function   Normalize features  Shadow mode

ANTI-PATTERN SUMMARY:
  Training/serving skew → one feature function for both
  Data leakage          → PIT join: feature_date < label_date always
  Irreproducibility     → log everything to MLflow
  Unsafe deployment     → model registry stages + A/B test
  Stale features        → materialisation SLA + online store refresh
```

---
*End of ML Pipeline Integration Master Guide — Sean Edition*
